# PS S6E4 - exp_20260409_024_xgb_digit_orderedte_min

目的:
 - yunsuxiaozi_xgb_lb0.98011 の骨格から、最小再現版を作る
 - 018 と異なる人格の候補として、XGB + digit + OrderedTE を確認する

今回残すもの:
 - digit features
 - frequency-based category remap
 - OrderedTE
 - XGBClassifier

今回切るもの:
 - Optuna class weight tuning
 - OrderedTE の4回増殖
 - 余計な周辺最適化

## Imports

In [1]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score
import torch

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

## CFG

In [2]:
class CFG:
    EXP_ID = "exp_20260409_024_xgb_digit_orderedte_min"
    EXP_NAME = "xgb_digit_orderedte_min"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 2026
    N_FOLDS = 5
    NUM_CLASSES = 3

    XGB_PARAMS = {
        "max_depth": 4,
        "colsample_bytree": 0.8,
        "subsample": 0.8,
        "n_estimators": 512,
        "learning_rate": 0.1,
        "early_stopping_rounds": 100,
        "random_state": SEED,
        "n_jobs": -1,
        "enable_categorical": True,
        "alpha": 5,
        "reg_lambda": 5,
        "max_leaves": 30,
        "min_child_weight": 2,
        "tree_method": "hist",
        "max_bin": 10000,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }

## utility

In [3]:
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)

def balanced_acc_score_mc(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    c = len(np.unique(y_true))
    acc = 0.0
    for i in range(c):
        acc += np.sum((y_true == i) & (y_pred == i)) / np.sum(y_true == i) / c
    return acc

seed_everything(CFG.SEED)

## load data

In [4]:
drop_cols = [CFG.ID_COL]

train = pd.read_csv(CFG.TRAIN_PATH).drop(drop_cols, axis=1)
test = pd.read_csv(CFG.TEST_PATH).drop(drop_cols, axis=1)

print(f"train.shape={train.shape}, test.shape={test.shape}")

target2idx = {v: i for i, v in enumerate(train[CFG.TARGET_COL].unique())}
idx2target = {v: k for k, v in target2idx.items()}

train[CFG.TARGET_COL] = train[CFG.TARGET_COL].map(target2idx)

CATS = [c for c in test.columns if train[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]

print("CATS:", CATS)
print("NUMS:", NUMS)
display(train.head())

train.shape=(630000, 20), test.shape=(270000, 19)
CATS: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
NUMS: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,0
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,0
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,0
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,1
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,0


## digit FE

In [5]:
M = train[NUMS].max()

def add_digit_features(df):
    df = df.copy()

    for c in NUMS:
        for k in range(-4, 4):
            df[f"{c}_digit{k}"] = (df[c] // (10 ** k) % 10).astype("int8")

        if M[c] < 10:
            df[c] = df[c].round(3)
        elif M[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)

    return df

train = add_digit_features(train)
test = add_digit_features(test)

DROP = [c for c in test.columns if test[c].nunique() == 1]
print("DROP:", DROP)

train.drop(DROP, axis=1, inplace=True)
test.drop(DROP, axis=1, inplace=True)

DROP: ['Soil_pH_digit1', 'Soil_pH_digit2', 'Soil_pH_digit3', 'Soil_Moisture_digit2', 'Soil_Moisture_digit3', 'Organic_Carbon_digit1', 'Organic_Carbon_digit2', 'Organic_Carbon_digit3', 'Electrical_Conductivity_digit1', 'Electrical_Conductivity_digit2', 'Electrical_Conductivity_digit3', 'Temperature_C_digit2', 'Temperature_C_digit3', 'Humidity_digit2', 'Humidity_digit3', 'Sunlight_Hours_digit2', 'Sunlight_Hours_digit3', 'Wind_Speed_kmh_digit2', 'Wind_Speed_kmh_digit3', 'Field_Area_hectare_digit2', 'Field_Area_hectare_digit3', 'Previous_Irrigation_mm_digit3']


## category remap

In [6]:
CATEGORY = CATS + [c for c in test.columns if "digit" in c]

for c in CATEGORY:
    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)

    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))

FEATURES = CATEGORY + NUMS

print("n_CATEGORY =", len(CATEGORY))
print("n_FEATURES =", len(FEATURES))
display(train[FEATURES + [CFG.TARGET_COL]].head())

n_CATEGORY = 74
n_FEATURES = 85


,Soil_Type,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Mulching_Used,Region,Soil_pH_digit-4,Soil_pH_digit-3,Soil_pH_digit-2,Soil_pH_digit-1,Soil_pH_digit0,Soil_Moisture_digit-4,Soil_Moisture_digit-3,Soil_Moisture_digit-2,Soil_Moisture_digit-1,Soil_Moisture_digit0,Soil_Moisture_digit1,Organic_Carbon_digit-4,Organic_Carbon_digit-3,Organic_Carbon_digit-2,Organic_Carbon_digit-1,Organic_Carbon_digit0,Electrical_Conductivity_digit-4,Electrical_Conductivity_digit-3,Electrical_Conductivity_digit-2,Electrical_Conductivity_digit-1,Electrical_Conductivity_digit0,Temperature_C_digit-4,Temperature_C_digit-3,Temperature_C_digit-2,Temperature_C_digit-1,Temperature_C_digit0,Temperature_C_digit1,Humidity_digit-4,Humidity_digit-3,Humidity_digit-2,Humidity_digit-1,Humidity_digit0,Humidity_digit1,Rainfall_mm_digit-4,Rainfall_mm_digit-3,Rainfall_mm_digit-2,Rainfall_mm_digit-1,Rainfall_mm_digit0,Rainfall_mm_digit1,Rainfall_mm_digit2,Rainfall_mm_digit3,Sunlight_Hours_digit-4,Sunlight_Hours_digit-3,Sunlight_Hours_digit-2,Sunlight_Hours_digit-1,Sunlight_Hours_digit0,Sunlight_Hours_digit1,Wind_Speed_kmh_digit-4,Wind_Speed_kmh_digit-3,Wind_Speed_kmh_digit-2,Wind_Speed_kmh_digit-1,Wind_Speed_kmh_digit0,Wind_Speed_kmh_digit1,Field_Area_hectare_digit-4,Field_Area_hectare_digit-3,Field_Area_hectare_digit-2,Field_Area_hectare_digit-1,Field_Area_hectare_digit0,Field_Area_hectare_digit1,Previous_Irrigation_mm_digit-4,Previous_Irrigation_mm_digit-3,Previous_Irrigation_mm_digit-2,Previous_Irrigation_mm_digit-1,Previous_Irrigation_mm_digit0,Previous_Irrigation_mm_digit1,Previous_Irrigation_mm_digit2,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,Previous_Irrigation_mm,Irrigation_Need
0,2,0,3,2,3,3,0,2,0,0,4,1,4,0,0,5,6,2,3,0,0,5,9,1,0,0,2,9,3,0,0,5,4,3,2,0,0,0,2,9,1,0,0,2,2,4,6,6,1,1,1,8,0,0,0,0,0,3,3,7,0,0,0,8,3,4,0,0,0,0,6,3,0,1,4.92,32.58,1.01,3.05,15.01,50.61,726.0,5.90,16.79,0.82,112.2,0
1,1,4,2,0,2,1,1,0,0,0,8,2,2,0,0,1,3,7,1,0,0,6,0,0,0,0,9,4,0,1,1,2,1,0,1,0,0,4,9,3,3,0,0,0,0,4,8,9,1,1,1,2,0,6,0,0,1,9,1,1,1,0,0,1,6,6,0,0,0,0,6,5,5,0,7.08,56.61,0.44,2.00,22.92,67.86,985.7,6.98,3.39,5.27,47.2,0
2,1,1,2,0,1,0,1,4,1,1,9,0,1,0,1,9,7,8,2,1,1,8,6,0,0,1,4,8,0,0,0,7,1,9,1,0,0,1,1,5,6,0,0,3,0,1,9,3,2,0,0,1,3,6,0,0,1,4,0,1,1,0,1,6,6,8,0,0,0,6,8,2,0,1,5.69,27.71,0.81,2.83,26.97,92.22,2201.7,6.05,3.85,8.24,110.4,0
3,0,4,1,0,0,1,1,0,1,1,5,0,1,0,1,2,0,5,4,1,1,6,2,1,0,0,5,8,2,0,1,2,6,8,2,0,0,9,6,0,3,0,0,9,4,3,3,2,0,0,0,4,5,2,0,0,1,0,1,5,1,0,1,5,4,8,0,0,1,0,9,4,3,0,5.65,13.32,1.33,0.87,13.32,61.57,1357.3,9.12,2.31,8.32,53.8,1
4,1,4,3,1,0,1,0,0,0,0,5,1,2,0,0,4,2,4,1,0,0,0,2,0,0,0,3,4,2,0,0,1,9,1,1,0,0,0,8,0,6,0,1,1,8,5,2,8,0,0,1,3,0,6,0,0,0,2,5,1,0,0,0,1,4,7,0,0,0,9,6,4,4,0,7.96,59.14,0.38,0.96,20.22,91.11,1538.2,6.95,13.94,7.37,93.2,0


## class weight

In [7]:
unique, counts = np.unique(train[CFG.TARGET_COL].values, return_counts=True)
count_dict = dict(zip(unique, counts))
avg_count = len(train) / len(unique)
weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
sample_weights = np.array([weights_dict[y] for y in train[CFG.TARGET_COL]])

print("weights_dict =", weights_dict)

weights_dict = {np.int64(0): np.float64(0.5676949153458749), np.int64(1): np.float64(0.8783891180136694), np.int64(2): np.float64(9.995716121662145)}


## OrderedTE class

In [8]:
class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train, category_cols=None, target_col="target"):
        if category_cols is None:
            category_cols = []

        self.train = train.copy()
        self.target_col = target_col
        self.category_cols = category_cols

        self.classes_ = sorted(train[target_col].unique())
        self.num_classes_ = len(self.classes_)
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values

        for c in self.category_cols:
            stats_list = []

            for k, cls in enumerate(self.classes_):
                y_binary = (train[target_col] == cls).astype(int)

                df = train[[c]].copy()
                df["y"] = y_binary.values
                df["cnt"] = 1

                df["cum_cnt"] = df.groupby(c)["cnt"].cumsum() - df["cnt"]
                df["cum_sum"] = df.groupby(c)["y"].cumsum() - df["y"]

                smooth_prior = self.a * self.global_prior_[k]

                te_col = f"{c}_TE_cls{cls}"
                df[te_col] = (df["cum_sum"] + smooth_prior) / (df["cum_cnt"] + self.a)
                df.loc[df["cum_cnt"] == 0, te_col] = self.global_prior_[k]

                self.train[te_col] = df[te_col].values

                stats_df = df.groupby(c)["y"].agg(["count", "sum"]).reset_index()
                stats_df.columns = [c, f"{c}_count_cls{cls}", f"{c}_sum_cls{cls}"]
                stats_df[f"{c}_prior_cls{cls}"] = self.global_prior_[k]
                stats_list.append(stats_df)

            combined_stats = stats_list[0]
            for i in range(1, len(stats_list)):
                combined_stats = combined_stats.merge(stats_list[i], on=c, how="outer")
            setattr(self, f"{c}_stats", combined_stats)

        return self.train

    def transform(self, test):
        test = test.copy()

        for c in self.category_cols:
            stats_df = getattr(self, f"{c}_stats")
            test = test.merge(stats_df, on=c, how="left")

            for k, cls in enumerate(self.classes_):
                te_col = f"{c}_TE_cls{cls}"
                count_col = f"{c}_count_cls{cls}"
                sum_col = f"{c}_sum_cls{cls}"
                prior_col = f"{c}_prior_cls{cls}"

                if count_col in test.columns:
                    test[te_col] = (test[sum_col] + self.a * test[prior_col]) / (test[count_col] + self.a)
                    test[te_col] = test[te_col].fillna(test[prior_col])
                    test.drop([count_col, sum_col, prior_col], axis=1, inplace=True)
                else:
                    test[te_col] = self.global_prior_[k]

        return test

## CV main

In [9]:
X = train.drop([CFG.TARGET_COL], axis=1)
y = train[CFG.TARGET_COL]
test_X = test.copy()

oof_preds = np.zeros((len(y), CFG.NUM_CLASSES))
test_preds = np.zeros((len(test_X), CFG.NUM_CLASSES))

kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=42)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    print(f"\nFold {fold}/{CFG.N_FOLDS}")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
    train_weights = sample_weights[train_idx]

    te = OrderedTE(a=1)
    full_df = pd.concat((X_train, y_train), axis=1)
    full_df["weight"] = train_weights

    # 共有コードでは4回増殖しているが、今回は最小再現なので1回だけ
    te_train = te.fit(full_df.sample(frac=1, random_state=42), category_cols=FEATURES, target_col=CFG.TARGET_COL)

    X_train = te_train.drop([CFG.TARGET_COL, "weight"], axis=1)
    y_train = te_train[CFG.TARGET_COL]
    train_weights = te_train["weight"].values

    X_val = te.transform(X_val)
    X_test = te.transform(test_X)

    # 元カテゴリは落とす
    X_train.drop(CATS, axis=1, inplace=True)
    X_val.drop(CATS, axis=1, inplace=True)
    X_test.drop(CATS, axis=1, inplace=True)

    params = CFG.XGB_PARAMS.copy()
    if not torch.cuda.is_available():
        params["device"] = "cpu"

    model = XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        sample_weight=train_weights,
        verbose=100,
    )

    y_pred = model.predict_proba(X_val)
    oof_preds[val_idx] = y_pred
    test_preds += model.predict_proba(X_test) / CFG.N_FOLDS

    fold_ba = balanced_acc_score_mc(y_val.values, y_pred)
    fold_scores.append(float(fold_ba))
    print(f"fold_ba = {fold_ba:.6f}")


Fold 1/5
[0]	validation_0-mlogloss:0.99806
[100]	validation_0-mlogloss:0.07776
[200]	validation_0-mlogloss:0.05922
[300]	validation_0-mlogloss:0.05416
[400]	validation_0-mlogloss:0.05145
[500]	validation_0-mlogloss:0.04992
[511]	validation_0-mlogloss:0.04976
fold_ba = 0.978039

Fold 2/5
[0]	validation_0-mlogloss:0.99739
[100]	validation_0-mlogloss:0.07597
[200]	validation_0-mlogloss:0.05815
[300]	validation_0-mlogloss:0.05349
[400]	validation_0-mlogloss:0.05133
[500]	validation_0-mlogloss:0.05015
[511]	validation_0-mlogloss:0.04998
fold_ba = 0.979457

Fold 3/5
[0]	validation_0-mlogloss:0.99933
[100]	validation_0-mlogloss:0.07640
[200]	validation_0-mlogloss:0.05843
[300]	validation_0-mlogloss:0.05346
[400]	validation_0-mlogloss:0.05135
[500]	validation_0-mlogloss:0.04984
[511]	validation_0-mlogloss:0.04969
fold_ba = 0.978011

Fold 4/5
[0]	validation_0-mlogloss:1.00050
[100]	validation_0-mlogloss:0.07695
[200]	validation_0-mlogloss:0.05887
[300]	validation_0-mlogloss:0.05367
[400]	valid

## raw CV

In [10]:
raw_cv_ba = balanced_acc_score_mc(y.values, oof_preds)
print(f"raw_cv_ba = {raw_cv_ba:.6f}")

raw_cv_ba = 0.978635


## simple bias search

In [11]:
best_score = raw_cv_ba
best_class_weights = np.ones(CFG.NUM_CLASSES, dtype=float)

grid = [
    [1.0, 1.0, 1.0],
    [1.2, 1.2, 1.2],
    [1.5, 1.5, 1.5],
    [1.5, 1.3, 1.8],
    [1.7, 1.5, 2.3],
    [1.8, 1.5, 2.4],
]

for cw in grid:
    cw = np.array(cw, dtype=float)
    adjusted = oof_preds * cw
    adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)
    score = balanced_acc_score_mc(y.values, np.argmax(adjusted, axis=1))
    if score > best_score:
        best_score = score
        best_class_weights = cw.copy()

print("best_class_weights =", best_class_weights.tolist())
print("biased_cv_ba =", best_score)
print("improvement =", best_score - raw_cv_ba)

best_class_weights = [1.8, 1.5, 2.4]
biased_cv_ba = 0.9791542562907178
improvement = 0.000518766015427996


## final prediction

In [12]:
final_test_probs = test_preds * best_class_weights
final_test_probs = final_test_probs / final_test_probs.sum(axis=1, keepdims=True)

final_test_preds = np.argmax(final_test_probs, axis=1)

submission = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")
submission[CFG.TARGET_COL] = final_test_preds
submission[CFG.TARGET_COL] = submission[CFG.TARGET_COL].map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof_preds)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", test_preds)

oof_biased = oof_preds * best_class_weights
oof_biased = oof_biased / oof_biased.sum(axis=1, keepdims=True)
pred_biased = final_test_probs.copy()

np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba_biased.npy", oof_biased)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba_biased.npy", pred_biased)

## save metadata

In [13]:
with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "raw_cv_ba": float(raw_cv_ba),
            "biased_cv_ba": float(best_score),
            "fold_scores": fold_scores,
            "best_class_weights": best_class_weights.tolist(),
            "n_features_before_te": int(len(FEATURES)),
            "n_features_after_te": int(X_train.shape[1]),
            "cats": CATS,
            "nums": NUMS,
            "category_cols": CATEGORY,
            "xgb_params": params,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

feature_cols_df = pd.DataFrame({"feature": X_train.columns.tolist()})
feature_cols_df.to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

print("saved to:", CFG.SAVE_DIR)
display(submission.head())

saved to: /kaggle/working/exp_20260409_024_xgb_digit_orderedte_min


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


## quick summary

In [14]:
summary_df = pd.DataFrame({
    "item": [
        "raw_cv_ba",
        "biased_cv_ba",
        "improvement",
        "n_features_before_te",
        "n_features_after_te",
    ],
    "value": [
        raw_cv_ba,
        best_score,
        best_score - raw_cv_ba,
        len(FEATURES),
        X_train.shape[1],
    ],
})
display(summary_df)

,item,value
0,raw_cv_ba,0.978635
1,biased_cv_ba,0.979154
2,improvement,0.000519
3,n_features_before_te,85.000000
4,n_features_after_te,332.000000
